# LeetCode 1219 — Path with Maximum Gold bằng Python

Link đề: https://leetcode.com/problems/path-with-maximum-gold/ · Độ khó: **Medium** · Chủ đề: Array, Backtracking, Matrix

Skeleton LeetCode:

```python
class Solution:
    def getMaximumGold(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: trả về **tổng vàng lớn nhất** thu được trên một đường đi đơn (không lặp ô) qua các ô có vàng, đi 4 hướng, bắt đầu và kết thúc ở đâu cũng được.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán dưới dạng **tìm đường đi đơn có trọng số lớn nhất** trên đồ thị lưới — một bài NP-khó trong tổng quát, nhưng ở đây nhỏ (≤ 25 ô có vàng) nên **quay lui (backtracking)** là đủ.
2. Cài đặt **DFS quay lui** trên lưới: đánh dấu ô đang thăm, thử 4 hướng, **khôi phục** (un-mark) khi quay về.
3. Biết hai cách đánh dấu: sửa tạm `grid[r][c] = 0` (in-place) hoặc dùng `visited`/bitmask; và bản **lặp** dùng stack tường minh.
4. Thêm **cắt tỉa (pruning)**: cận trên "vàng còn lại", và dừng sớm khi đã gom được toàn bộ vàng.
5. Tránh các lỗi: quên khôi phục ô, chỉ bắt đầu từ `(0, 0)`, nghĩ rằng tham lam (đi sang ô nhiều vàng nhất) là đủ, lấy `max` chỉ trên ô kết thúc.

## 2. Đề bài & ví dụ

Cho mỏ vàng là lưới `grid` cỡ `m × n`, mỗi ô chứa số vàng (0 nghĩa là ô trống). Trả về số vàng **lớn nhất** có thể thu được với các luật:

- Mỗi khi đứng ở một ô, bạn lấy **toàn bộ** vàng ô đó.
- Từ vị trí hiện tại, bạn đi **một bước** sang trái, phải, lên hoặc xuống.
- **Không** được thăm một ô quá một lần.
- **Không bao giờ** đi vào ô có `0` vàng.
- Có thể **bắt đầu và dừng** ở bất kỳ ô nào có vàng.

**Ví dụ 1:** `grid = [[0,6,0],[5,8,7],[0,9,0]]` → `24` (đường `9 → 8 → 7`).

**Ví dụ 2:** `grid = [[1,0,7],[2,0,6],[3,4,5],[0,3,0],[9,0,20]]` → `28` (đường `1 → 2 → 3 → 4 → 5 → 6 → 7`).

**Ràng buộc:**
- `1 <= m, n <= 15`
- `0 <= grid[i][j] <= 100`
- Có **tối đa 25** ô chứa vàng.

In [1]:
import random
import time
from itertools import permutations
from typing import List

examples = [
    ([[0, 6, 0], [5, 8, 7], [0, 9, 0]], 24),
    ([[1, 0, 7], [2, 0, 6], [3, 4, 5], [0, 3, 0], [9, 0, 20]], 28),
]
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
for g, want in examples:
    for row in g:
        print(row)
    print("->", want, "\n")

[0, 6, 0]
[5, 8, 7]
[0, 9, 0]
-> 24 

[1, 0, 7]
[2, 0, 6]
[3, 4, 5]
[0, 3, 0]
[9, 0, 20]
-> 28 



## 3. Kiến thức nền

### 3.1 Duyệt 4 hàng xóm trên lưới

Ô `(r, c)` có tối đa 4 hàng xóm `(r ± 1, c)` và `(r, c ± 1)`. Phải kiểm tra **trong biên** `0 <= x < m`, `0 <= y < n` trước khi truy cập `grid[x][y]` (chỉ số âm trong Python không báo lỗi mà lấy từ cuối list — rất nguy hiểm!).

In [2]:
def neighbors(grid, r, c):
    m, n = len(grid), len(grid[0])
    for dr, dc in DIRS:
        x, y = r + dr, c + dc
        if 0 <= x < m and 0 <= y < n:
            yield x, y


g = examples[0][0]
print("hàng xóm của (0,0):", list(neighbors(g, 0, 0)))
print("hàng xóm của (1,1):", list(neighbors(g, 1, 1)))
print("cảnh báo: g[-1][0] =", g[-1][0], "(không lỗi, nhưng là hàng cuối!)")

hàng xóm của (0,0): [(1, 0), (0, 1)]
hàng xóm của (1,1): [(2, 1), (0, 1), (1, 2), (1, 0)]
cảnh báo: g[-1][0] = 0 (không lỗi, nhưng là hàng cuối!)


### 3.2 Đệ quy và quay lui (backtracking)

**Quay lui** = thử một lựa chọn, đi sâu tiếp, rồi **hoàn tác** lựa chọn để thử lựa chọn khác. Khuôn mẫu:

```
def backtrack(state):
    if không còn lựa chọn: ghi nhận
    for lựa chọn in các lựa chọn hợp lệ:
        làm lựa chọn        # đánh dấu
        backtrack(state mới)
        hoàn tác lựa chọn   # bỏ đánh dấu
```

Ví dụ nhỏ: liệt kê mọi đường đơn bắt đầu từ ô `(1,1)` trong ví dụ 1.

In [3]:
def all_paths_from(grid, r, c):
    paths, path, seen = [], [(r, c)], {(r, c)}

    def go(r, c):
        paths.append(list(path))
        for x, y in neighbors(grid, r, c):
            if grid[x][y] and (x, y) not in seen:
                seen.add((x, y)); path.append((x, y))      # làm lựa chọn
                go(x, y)
                seen.remove((x, y)); path.pop()             # hoàn tác

    go(r, c)
    return paths


ps = all_paths_from(examples[0][0], 1, 1)
print(len(ps), "đường đơn bắt đầu tại (1,1), ví dụ:", ps[:4])

5 đường đơn bắt đầu tại (1,1), ví dụ: [[(1, 1)], [(1, 1), (2, 1)], [(1, 1), (0, 1)], [(1, 1), (1, 2)]]


### 3.3 Bitmask để đánh dấu tập ô

Vì có ≤ 25 ô vàng, ta đánh số chúng `0..k−1` và biểu diễn tập ô đã thăm bằng **một số nguyên** `mask`: bit thứ `i` bật nghĩa là ô `i` đã thăm.
- Kiểm tra: `mask >> i & 1`.
- Bật bit: `mask | (1 << i)`.

In [4]:
mask = 0
mask |= 1 << 3
mask |= 1 << 0
print(bin(mask), [i for i in range(5) if mask >> i & 1])

0b1001 [0, 3]


## 4. Ý tưởng nền tảng / trực giác

Xem các ô có vàng là **đỉnh** của đồ thị, hai ô kề nhau (4 hướng) có **cạnh**. Ta cần đường đi **đơn** (không lặp đỉnh) có tổng trọng số lớn nhất.

Ví dụ 1: đồ thị là hình chữ thập — tâm `8` nối với `6, 5, 7, 9`; các "cánh" không nối nhau. Mọi đường đơn đi qua tâm tối đa nối được **hai** cánh: `cánh_a → 8 → cánh_b`. Chọn hai cánh lớn nhất `9` và `7`: `9 + 8 + 7 = 24`.

Tại sao không tham lam "đi sang hàng xóm nhiều vàng nhất"? Ví dụ 2 cho thấy: bắt đầu ở `20` thì không đi đâu được (hàng xóm đều 0) → chỉ 20; bắt đầu ở `9` cũng vậy. Đường tốt nhất `1→2→3→4→5→6→7 = 28` đi qua toàn ô nhỏ. Vì phải **thử nhiều khả năng** và số ô vàng nhỏ (≤ 25) → quay lui.

In [5]:
g = examples[0][0]
gold_cells = [(r, c) for r in range(3) for c in range(3) if g[r][c]]
print("ô có vàng:", gold_cells)
print("cạnh:", [((r, c), (x, y)) for r, c in gold_cells for x, y in neighbors(g, r, c) if g[x][y] and (r, c) < (x, y)])

ô có vàng: [(0, 1), (1, 0), (1, 1), (1, 2), (2, 1)]
cạnh: [((0, 1), (1, 1)), ((1, 0), (1, 1)), ((1, 1), (2, 1)), ((1, 1), (1, 2))]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng "ngây thơ nhất":** mọi đường đơn là một **hoán vị** của một tập con các ô vàng. Thử mọi tập con, mọi thứ tự; giữ những dãy mà hai ô liên tiếp kề nhau; lấy tổng lớn nhất.

Số khả năng `Σ k!/(k−j)!` — bùng nổ cực nhanh, chỉ dùng cho `k ≤ 7`. Đây là **trọng tài** để kiểm tra các cách sau.

In [6]:
def gold_brute(grid):
    cells = [(r, c) for r in range(len(grid)) for c in range(len(grid[0])) if grid[r][c]]
    best = 0
    for length in range(1, len(cells) + 1):
        for perm in permutations(cells, length):
            if all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 for a, b in zip(perm, perm[1:])):
                best = max(best, sum(grid[r][c] for r, c in perm))
    return best


print(gold_brute(examples[0][0]))
assert gold_brute(examples[0][0]) == 24

24


Ví dụ 2 có 11 ô vàng → `11!` ≈ 40 triệu hoán vị dài nhất — quá chậm cho brute force này.

**Độ phức tạp:** Time `O(k! · k)` với `k` = số ô vàng, Space `O(k)`.

**Nút thắt:** ta sinh cả những dãy **không thể** là đường đi (hai ô liên tiếp không kề nhau) rồi mới loại. Tốt hơn: chỉ mở rộng đường đi bằng **hàng xóm** của ô cuối → chính là DFS quay lui.

## 6. Bước đột phá

**Insight 1:** chỉ sinh **đường đi hợp lệ** bằng cách mở rộng từ ô hiện tại sang hàng xóm có vàng, chưa thăm. Mỗi đường được sinh **đúng một lần** (theo ô bắt đầu và chuỗi lựa chọn).

**Insight 2 (giá trị trả về):** gọi `dfs(r, c)` = vàng lớn nhất thu được trên một đường **bắt đầu tại `(r, c)`**, khi các ô đang nằm trên đường hiện tại bị cấm. Khi đó:

`dfs(r, c) = grid[r][c] + max(0, max(dfs(x, y) : (x, y) là hàng xóm có vàng, chưa thăm))`

**Quy tắc quay lui:**
- **Nếu** bước vào `(r, c)` **thì** đánh dấu nó (đặt tạm `grid[r][c] = 0` — vừa "đã thăm" vừa "không có vàng" nên tự động không quay lại).
- **Khi** thử xong mọi hướng **thì** khôi phục `grid[r][c]` về giá trị cũ, để các đường khác vẫn dùng được ô này.

Vì điểm bắt đầu tuỳ ý → chạy `dfs` từ **mọi** ô có vàng, lấy max.

**Cắt tỉa:** tổng vàng còn chưa dùng là **cận trên** cho phần còn lại; nếu `đã có + còn lại ≤ best` thì nhánh này vô vọng. Và nếu `best == tổng toàn bộ vàng` thì không thể tốt hơn → dừng hẳn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS quay lui, đánh dấu tại chỗ

In [7]:
def gold_backtrack(grid):
    grid = [row[:] for row in grid]          # không làm hỏng input của người gọi
    m, n = len(grid), len(grid[0])

    def dfs(r, c):
        g = grid[r][c]
        grid[r][c] = 0                       # đánh dấu đã thăm
        best = 0
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and grid[x][y]:
                best = max(best, dfs(x, y))
        grid[r][c] = g                       # khôi phục
        return g + best

    return max((dfs(r, c) for r in range(m) for c in range(n) if grid[r][c]), default=0)


for g, want in examples:
    got = gold_backtrack(g)
    print(got)
    assert got == want

24
28


**Độ phức tạp:** mỗi bước trong đường đi có tối đa **3** hướng mới (không quay lại ô vừa đến), nên từ một ô bắt đầu có tối đa `O(4 · 3^(k−1))` đường. Tổng `O(k · 3^k)` trong trường hợp xấu nhất lý thuyết; thực tế trên lưới ít hơn rất nhiều. Space `O(k)` cho ngăn xếp đệ quy (độ sâu ≤ 25 — an toàn).

### 7.2 Cách 3 — Đồ thị ô vàng + bitmask, DFS lặp bằng stack

Không sửa lưới; đánh số các ô vàng, dựng danh sách kề, và DFS **không đệ quy** với trạng thái `(ô hiện tại, mask đã thăm, tổng hiện tại)`.

In [8]:
def build_gold_graph(grid):
    m, n = len(grid), len(grid[0])
    cells = [(r, c) for r in range(m) for c in range(n) if grid[r][c]]
    index = {cell: i for i, cell in enumerate(cells)}
    adj = [[] for _ in cells]
    for i, (r, c) in enumerate(cells):
        for x, y in neighbors(grid, r, c):
            if (x, y) in index:
                adj[i].append(index[(x, y)])
    values = [grid[r][c] for r, c in cells]
    return values, adj


def gold_stack_bitmask(grid):
    values, adj = build_gold_graph(grid)
    best = 0
    for s in range(len(values)):
        stack = [(s, 1 << s, values[s])]
        while stack:
            u, mask, total = stack.pop()
            if total > best:
                best = total
            for v in adj[u]:
                if not mask >> v & 1:
                    stack.append((v, mask | (1 << v), total + values[v]))
    return best


vals, adj = build_gold_graph(examples[0][0])
print("giá trị:", vals, "| kề:", adj)
for g, want in examples:
    assert gold_stack_bitmask(g) == want
print("Cách 3 đúng trên ví dụ")

giá trị: [6, 5, 8, 7, 9] | kề: [[2], [2], [4, 0, 3, 1], [2], [2]]
Cách 3 đúng trên ví dụ


**Độ phức tạp:** như Cách 2 về số trạng thái, nhưng mỗi trạng thái là một tuple trên stack. Không có đệ quy, không sửa input. Space tối đa `O(k · 3)` phần tử trên stack mỗi tầng → `O(k²)` tổng cộng trong DFS lặp.

### 7.3 Cách 4 — Quay lui + cắt tỉa

Thêm hai kỹ thuật:
1. **Dừng sớm toàn cục:** nếu `best == total_gold` (đã gom hết vàng), không thể tốt hơn → thoát.
2. **Cận trên:** truyền vào `remaining` = vàng chưa nằm trên đường hiện tại. Nếu `current + remaining <= best` thì bỏ nhánh.

In [9]:
def gold_pruned(grid):
    grid = [row[:] for row in grid]
    m, n = len(grid), len(grid[0])
    total_gold = sum(map(sum, grid))
    best = 0

    def dfs(r, c, current, remaining):
        nonlocal best
        g = grid[r][c]
        current += g
        remaining -= g
        if current > best:
            best = current
        if best == total_gold or current + remaining <= best:
            return                              # cắt tỉa
        grid[r][c] = 0
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and grid[x][y]:
                dfs(x, y, current, remaining)
                if best == total_gold:
                    break
        grid[r][c] = g

    for r in range(m):
        for c in range(n):
            if grid[r][c] and best < total_gold:
                dfs(r, c, 0, total_gold)
    return best


for g, want in examples:
    assert gold_pruned(g) == want
print("Cách 4 đúng trên ví dụ")

Cách 4 đúng trên ví dụ


Lưu ý `remaining` ở đây là cận trên **thô** (cả những ô không còn với tới được). Vẫn đúng vì nó **không nhỏ hơn** phần vàng thật sự còn lấy được. Cận càng chặt cắt càng nhiều; cận thô thì cắt ít nhưng rẻ.

Hiệu quả rõ nhất trên lưới **đầy vàng** (ví dụ `5 × 5` toàn ô 1): có đường Hamilton đi qua cả 25 ô, nên ngay khi tìm thấy, mọi thứ dừng lại.

In [10]:
full = [[1] * 5 for _ in range(5)]
for f in [gold_backtrack, gold_pruned]:
    t0 = time.perf_counter()
    r = f(full)
    print(f"{f.__name__:15s} -> {r} trong {time.perf_counter() - t0:.3f}s")

gold_backtrack  -> 25 trong 1.491s
gold_pruned     -> 25 trong 0.000s


## 8. TRACE từng bước

Trace Cách 2 với ô bắt đầu `(2,1)` (giá trị 9) trong ví dụ 1. Mỗi dòng in độ sâu, ô đang vào, và kết quả trả về.

| Gọi | Ô | Hàng xóm thử | Trả về |
|---|---|---|---|
| dfs(2,1) | 9 | (1,1) | 9 + dfs(1,1) |
| └ dfs(1,1) | 8 | (0,1),(1,0),(1,2) — (2,1) đang bị đánh dấu 0 | 8 + max(6, 5, 7) = 15 |
| &nbsp;&nbsp;├ dfs(0,1) | 6 | không | 6 |
| &nbsp;&nbsp;├ dfs(1,0) | 5 | không | 5 |
| &nbsp;&nbsp;└ dfs(1,2) | 7 | không | 7 |
| kết quả | | | 9 + 15 = **24** |

In [11]:
def trace_dfs(grid, sr, sc):
    grid = [row[:] for row in grid]
    m, n = len(grid), len(grid[0])

    def dfs(r, c, depth):
        g = grid[r][c]
        print("  " * depth + f"vào ({r},{c}) vàng={g}, lưới đánh dấu ô này = 0")
        grid[r][c] = 0
        best = 0
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and grid[x][y]:
                best = max(best, dfs(x, y, depth + 1))
        grid[r][c] = g
        print("  " * depth + f"rời ({r},{c}) khôi phục {g}, trả về {g} + {best} = {g + best}")
        return g + best

    return dfs(sr, sc, 0)


print("Kết quả:", trace_dfs(examples[0][0], 2, 1))

vào (2,1) vàng=9, lưới đánh dấu ô này = 0
  vào (1,1) vàng=8, lưới đánh dấu ô này = 0
    vào (0,1) vàng=6, lưới đánh dấu ô này = 0
    rời (0,1) khôi phục 6, trả về 6 + 0 = 6
    vào (1,2) vàng=7, lưới đánh dấu ô này = 0
    rời (1,2) khôi phục 7, trả về 7 + 0 = 7
    vào (1,0) vàng=5, lưới đánh dấu ô này = 0
    rời (1,0) khôi phục 5, trả về 5 + 0 = 5
  rời (1,1) khôi phục 8, trả về 8 + 7 = 15
rời (2,1) khôi phục 9, trả về 9 + 15 = 24
Kết quả: 24


Trace thêm **từng ô bắt đầu** để thấy vì sao phải thử tất cả:

In [12]:
def trace_starts(grid):
    m, n = len(grid), len(grid[0])
    g2 = [row[:] for row in grid]

    def dfs(r, c):
        g = g2[r][c]
        g2[r][c] = 0
        best = 0
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and g2[x][y]:
                best = max(best, dfs(x, y))
        g2[r][c] = g
        return g + best

    ans = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c]:
                v = dfs(r, c)
                ans = max(ans, v)
                print(f"bắt đầu ({r},{c}) vàng={grid[r][c]:>2}: tốt nhất {v:>2} | best toàn cục {ans}")
    return ans


trace_starts(examples[1][0])

bắt đầu (0,0) vàng= 1: tốt nhất 28 | best toàn cục 28
bắt đầu (0,2) vàng= 7: tốt nhất 28 | best toàn cục 28
bắt đầu (1,0) vàng= 2: tốt nhất 27 | best toàn cục 28
bắt đầu (1,2) vàng= 6: tốt nhất 21 | best toàn cục 28
bắt đầu (2,0) vàng= 3: tốt nhất 25 | best toàn cục 28
bắt đầu (2,1) vàng= 4: tốt nhất 22 | best toàn cục 28
bắt đầu (2,2) vàng= 5: tốt nhất 18 | best toàn cục 28
bắt đầu (3,1) vàng= 3: tốt nhất 25 | best toàn cục 28
bắt đầu (4,0) vàng= 9: tốt nhất  9 | best toàn cục 28
bắt đầu (4,2) vàng=20: tốt nhất 20 | best toàn cục 28


28

## 9. Code nộp LeetCode

Dùng quay lui đánh dấu tại chỗ (Cách 2) kèm dừng sớm khi đã gom hết vàng (ý từ Cách 4) — ngắn gọn và đủ nhanh cho mọi test.

In [13]:
class Solution:
    def getMaximumGold(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        total = sum(map(sum, grid))

        def dfs(r: int, c: int) -> int:
            g = grid[r][c]
            grid[r][c] = 0
            best = 0
            for x, y in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= x < m and 0 <= y < n and grid[x][y]:
                    v = dfs(x, y)
                    if v > best:
                        best = v
            grid[r][c] = g
            return g + best

        ans = 0
        for r in range(m):
            for c in range(n):
                if grid[r][c]:
                    ans = max(ans, dfs(r, c))
                    if ans == total:
                        return ans
        return ans


sol = Solution()
for g, want in examples:
    got = sol.getMaximumGold([row[:] for row in g])
    print(got)
    assert got == want

24
28


## 10. Vì sao đúng

1. **Đầy đủ:** mọi đường đi hợp lệ `p0 → p1 → ... → pt` được duyệt: vòng ngoài thử `p0`; tại `pi`, ô `p(i+1)` là hàng xóm có vàng và chưa nằm trên đường (vì các ô trên đường đang bị đặt `0`), nên `dfs` sẽ thử nó.
2. **Hợp lệ:** `dfs` chỉ đi sang ô kề, có vàng (`grid[x][y] != 0`), và ô trên đường hiện tại bị đặt `0` nên không lặp ô và không vào ô trống.
3. **Khôi phục đúng:** mỗi lần vào ô ta lưu `g` và gán lại khi rời → sau khi `dfs(r, c)` kết thúc, lưới trở về y như trước khi gọi. Nhờ đó các nhánh/ô bắt đầu khác thấy lưới nguyên vẹn.
4. **Giá trị:** `dfs(r, c)` trả về `grid[r][c] +` giá trị tốt nhất của phần đường còn lại (hoặc 0 nếu dừng ngay) — đúng định nghĩa ở mục 6, chứng minh bằng quy nạp theo số ô còn có thể thăm.
5. **Dừng sớm:** không đường nào thu được nhiều hơn tổng toàn bộ vàng, nên khi `ans == total` thì `ans` là tối ưu. Cắt tỉa cận trên ở Cách 4 cũng đúng vì `remaining` ≥ vàng thật sự còn lấy được.

## 11. Time / Space Complexity

Gọi `k` = số ô có vàng (≤ 25), `m·n` = kích thước lưới (≤ 225).

| Cách | Time (xấu nhất) | Space |
|---|---|---|
| 1. Hoán vị | `O(k! · k)` | `O(k)` |
| 2. Quay lui tại chỗ | `O(m·n + k · 3^k)` | `O(k)` đệ quy |
| 3. Bitmask + stack | `O(k · 3^k)` | `O(k²)` stack |
| 4. Quay lui + cắt tỉa | cùng cận trên, thực tế nhanh hơn nhiều | `O(k)` |

`3^k` là cận lỏng: trên lưới, các ô vàng bị giới hạn hình học nên số đường đơn thực tế nhỏ hơn rất nhiều (lưới `5×5` đầy cũng chỉ mất ~1–2 giây với Cách 2 trong Python).

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [14]:
methods = {
    "backtrack": gold_backtrack,
    "stack_bitmask": gold_stack_bitmask,
    "pruned": gold_pruned,
    "submit": lambda g: Solution().getMaximumGold([row[:] for row in g]),
}
edge = [
    ([[0]], 0),                              # không có vàng
    ([[7]], 7),                              # một ô
    ([[1, 0, 1]], 1),                        # hai ô không nối nhau
    ([[1, 2, 3, 4]], 10),                    # một hàng
    ([[100] * 15], 1500),
    ([[1, 1], [1, 1]], 4),                   # chu trình 4 ô
    ([[0, 0, 0], [0, 50, 0], [0, 0, 0]], 50),
]
for name, f in methods.items():
    for g, want in examples + edge:
        assert f(g) == want, (name, g)
# input không bị thay đổi
g = [row[:] for row in examples[1][0]]
Solution().getMaximumGold(g)
assert g == examples[1][0]
print("Mọi cách đúng trên ví dụ + edge case; lưới được khôi phục nguyên vẹn")

Mọi cách đúng trên ví dụ + edge case; lưới được khôi phục nguyên vẹn


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [15]:
rng = random.Random(1219)
cnt = 0
while cnt < 150:
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    g = [[rng.choice([0, 0, rng.randint(1, 9)]) for _ in range(n)] for _ in range(m)]
    if sum(v > 0 for row in g for v in row) > 7:
        continue
    want = gold_brute(g)
    for name, f in methods.items():
        assert f(g) == want, (name, g)
    cnt += 1
print("150 lưới ngẫu nhiên (≤ 7 ô vàng): mọi cách khớp brute force")

for _ in range(150):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.choice([0, rng.randint(1, 100)]) for _ in range(n)] for _ in range(m)]
    want = gold_backtrack(g)
    for name in ["stack_bitmask", "pruned", "submit"]:
        assert methods[name](g) == want, name
print("150 lưới lớn hơn: các cách khớp nhau")

150 lưới ngẫu nhiên (≤ 7 ô vàng): mọi cách khớp brute force
150 lưới lớn hơn: các cách khớp nhau


### 12.3 Test giới hạn (25 ô vàng)

In [16]:
heavy = [
    [[1] * 5 for _ in range(5)],                                         # 5x5 đầy
    [[1, 1, 1, 1, 1], [1, 0, 1, 0, 1], [1, 1, 1, 1, 1], [1, 0, 1, 0, 1], [1, 1, 1, 1, 1]],
    [[rng.randint(1, 100) if (r + c) % 3 else 0 for c in range(15)] for r in range(2)],
]
for g in heavy:
    t0 = time.perf_counter()
    r_sub = methods["submit"](g)
    r_pr = methods["pruned"](g)
    dt = time.perf_counter() - t0
    assert r_sub == r_pr
    print(f"{len(g)}x{len(g[0])}: {r_sub} trong {dt:.3f}s")

# lưới 5x5 giá trị ngẫu nhiên — chạy bản tại chỗ KHÔNG có dừng sớm một lần để thấy chi phí thật
g = [[rng.randint(1, 100) for _ in range(5)] for _ in range(5)]
t0 = time.perf_counter()
r = gold_backtrack(g)
print("5x5 ngẫu nhiên:", r, f"trong {time.perf_counter() - t0:.2f}s", "| tổng toàn lưới:", sum(map(sum, g)))

5x5: 25 trong 0.084s
5x5: 19 trong 0.005s
2x15: 237 trong 0.000s


5x5 ngẫu nhiên: 987 trong 1.46s | tổng toàn lưới: 987


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên khôi phục ô sau khi quay lui

In [17]:
def wrong_no_restore(grid):
    grid = [row[:] for row in grid]
    m, n = len(grid), len(grid[0])

    def dfs(r, c):
        g = grid[r][c]
        grid[r][c] = 0
        best = 0
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and grid[x][y]:
                best = max(best, dfs(x, y))
        return g + best                      # quên grid[r][c] = g

    return max((dfs(r, c) for r in range(m) for c in range(n) if grid[r][c]), default=0)


for g, want in examples:
    print("sai:", wrong_no_restore(g), "| đúng:", want)

sai: 23 | đúng: 24
sai: 28 | đúng: 28


**Vì sao sai:** ô bị "ăn" vĩnh viễn — các nhánh khác và các ô bắt đầu khác không còn dùng được ô đó. Ở ví dụ 1, đường từ `(0,1)` ăn mất tâm `8` trước khi kịp thử đường `9 → 8 → 7`. **Sửa:** khôi phục `grid[r][c] = g` trước khi `return`.

### Lỗi 2 — Chỉ bắt đầu từ `(0, 0)` hoặc từ ô có vàng lớn nhất

In [18]:
def wrong_start_max_cell(grid):
    m, n = len(grid), len(grid[0])
    r, c = max(((r, c) for r in range(m) for c in range(n)), key=lambda p: grid[p[0]][p[1]])
    g2 = [row[:] for row in grid]

    def dfs(r, c):
        g = g2[r][c]
        g2[r][c] = 0
        best = 0
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and g2[x][y]:
                best = max(best, dfs(x, y))
        g2[r][c] = g
        return g + best

    return dfs(r, c)


print("sai :", wrong_start_max_cell(examples[1][0]))   # bắt đầu ở 20 -> chỉ 20
print("đúng:", gold_backtrack(examples[1][0]))

sai : 20
đúng: 28


**Vì sao sai:** đề cho phép bắt đầu ở **bất kỳ** ô có vàng; ô lớn nhất có thể bị cô lập. **Sửa:** thử mọi ô có vàng làm điểm đầu.

### Lỗi 3 — Tham lam: luôn đi sang hàng xóm nhiều vàng nhất

In [19]:
def wrong_greedy(grid):
    m, n = len(grid), len(grid[0])
    ans = 0
    for sr in range(m):
        for sc in range(n):
            if not grid[sr][sc]:
                continue
            seen, r, c, total = {(sr, sc)}, sr, sc, grid[sr][sc]
            while True:
                cand = [(grid[x][y], x, y) for x, y in neighbors(grid, r, c) if grid[x][y] and (x, y) not in seen]
                if not cand:
                    break
                _, r, c = max(cand)
                seen.add((r, c))
                total += grid[r][c]
            ans = max(ans, total)
    return ans


g = [[1] * 11, [0] * 5 + [4] + [0] * 5]
for row in g:
    print(row)
print("sai :", wrong_greedy(g))      # tới ô giữa thì nhảy xuống 4 (ngõ cụt)
print("đúng:", gold_backtrack(g))

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0]
sai : 10
đúng: 11


**Vì sao sai:** chọn ô lớn trước có thể "chặn đường" đến nhiều ô khác. Ở lưới chữ T trên, đường tốt nhất là cả hàng trên (`11`); tham lam đến ô giữa thì thấy `4 > 1` nên rẽ xuống ngõ cụt, chỉ được `10` — dù bắt đầu ở đâu. **Sửa:** quay lui thử mọi hướng.

### Lỗi 4 — Lấy max chỉ khi "không đi tiếp được" nhưng quên cộng ô hiện tại / quên trường hợp không có vàng

Viết `return max(dfs(...) for ...)` trên danh sách rỗng gây `ValueError`; còn `max(...)` cho cả lưới không có vàng cũng lỗi. Luôn khởi tạo `best = 0` và dùng `default=0`.

In [20]:
try:
    print(max(v for v in []))
except ValueError as e:
    print("max() trên rỗng:", e)
print("với default:", max((v for v in []), default=0))
print("lưới không có vàng:", Solution().getMaximumGold([[0, 0], [0, 0]]))

max() trên rỗng: max() iterable argument is empty
với default: 0
lưới không có vàng: 0


## 14. Mẹo & nhận dạng pattern

- "Đường đi **không lặp ô**, tối đa/đếm, lưới nhỏ" → **quay lui DFS** (giống Word Search — LeetCode 79, Unique Paths III — LeetCode 980).
- Ràng buộc "tối đa 25 ô" là gợi ý rõ: lời giải hàm mũ theo số ô là chấp nhận được.
- Đánh dấu tại chỗ bằng cách gán giá trị "cấm" (ở đây là `0`, trùng với "ô trống") giúp code ngắn và không tốn bộ nhớ.
- Luôn hỏi: "trạng thái phải **khôi phục** khi quay về không?" — gần như mọi bug quay lui nằm ở đây.
- Cắt tỉa bằng **cận trên** và **dừng khi đạt tối ưu tuyệt đối**.

## 15. Biến thể

1. **Trả về chính đường đi** tốt nhất (danh sách ô): lưu `path` song song, cập nhật khi tốt hơn.
2. **Đi 8 hướng** (thêm đường chéo): chỉ đổi `DIRS`.
3. **Bắt buộc bắt đầu tại ô cho trước** và kết thúc tại ô cho trước → giống Unique Paths III.
4. **Giới hạn số bước tối đa `L`**: thêm tham số độ sâu, dừng khi đạt `L`.
5. **Lưới lớn, nhiều ô** → không quay lui được nữa; bài toán đường đi đơn dài nhất là NP-khó, phải dùng heuristic hoặc cấu trúc đặc biệt (ví dụ cây → DP trên cây).

In [21]:
def best_path(grid):
    """Biến thể 1: trả về (tổng, đường đi)."""
    grid = [row[:] for row in grid]
    m, n = len(grid), len(grid[0])
    best = (0, [])
    path = []

    def dfs(r, c, total):
        nonlocal best
        g = grid[r][c]
        grid[r][c] = 0
        path.append((r, c))
        total += g
        if total > best[0]:
            best = (total, list(path))
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and grid[x][y]:
                dfs(x, y, total)
        path.pop()
        grid[r][c] = g

    for r in range(m):
        for c in range(n):
            if grid[r][c]:
                dfs(r, c, 0)
    return best


for g, _ in examples:
    total, p = best_path(g)
    print(total, [g[r][c] for r, c in p])

24 [7, 8, 9]
28 [1, 2, 3, 4, 5, 6, 7]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | thử mọi thứ tự tập con | `O(k!·k)` | `O(k)` | trọng tài, `k ≤ 7` |
| 2. Quay lui tại chỗ | DFS, gán 0 rồi khôi phục | `O(k·3^k)` | `O(k)` | **nộp bài**, ngắn gọn |
| 3. Bitmask + stack | đồ thị ô vàng, DFS lặp | `O(k·3^k)` | `O(k²)` | không muốn đệ quy / sửa input |
| 4. Quay lui + cắt tỉa | cận trên + dừng khi đạt tổng | thực tế nhanh nhất | `O(k)` | test "lưới đầy" |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`Solution().getMaximumGold([[1, 2], [0, 3]])` = ?

### Bài 2 (tính tay)

Lưới `[[5, 0, 3], [4, 1, 2], [0, 6, 0]]`: liệt kê đồ thị ô vàng (ai kề ai) và tìm đường tốt nhất.

### Bài 3 (điền TODO) — đếm số đường đi đơn

In [22]:
def count_paths(grid):
    """Đếm số đường đi đơn (có hướng, độ dài >= 1 ô) qua các ô có vàng."""
    grid = [row[:] for row in grid]
    m, n = len(grid), len(grid[0])

    def dfs(r, c):
        # TODO: đánh dấu, đếm 1 (đường dừng tại đây) + tổng số đường tiếp tục từ các hàng xóm, khôi phục
        raise NotImplementedError

    return sum(dfs(r, c) for r in range(m) for c in range(n) if grid[r][c])

# print(count_paths([[1, 1]]))   # mong đợi 4: (A), (B), (A->B), (B->A)

### Bài 4 (tự viết lại)

Viết lại lời giải dùng tập `visited` (set các tuple) thay vì sửa `grid`.

In [23]:
def gold_visited_todo(grid):
    # TODO
    raise NotImplementedError

# print(gold_visited_todo([[0, 6, 0], [5, 8, 7], [0, 9, 0]]))   # mong đợi 24

### Bài 5 (mini challenge) — giới hạn số bước

Tìm vàng lớn nhất khi đường đi gồm **tối đa `L` ô**.

In [24]:
def gold_limited_todo(grid, L):
    # TODO
    raise NotImplementedError

# print(gold_limited_todo([[0, 6, 0], [5, 8, 7], [0, 9, 0]], 2))   # mong đợi 17 (9 + 8)
# print(gold_limited_todo([[1, 0, 7], [2, 0, 6], [3, 4, 5], [0, 3, 0], [9, 0, 20]], 3))   # mong đợi 20 (ô 20 đứng một mình!)

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: ba ô vàng `1, 2, 3` nối thành chữ L. Mức 2: đi qua cả ba được không? Mức 3: `1 → 2 → 3` = 6.

**Bài 2.** Mức 1: `5–4`, `4–1`, `1–2`, `1–6`, `3–2`. Mức 2: `1` có ba hàng xóm `4, 2, 6` nhưng đường đơn qua `1` chỉ dùng được hai. Mức 3: `5 → 4 → 1 → 2 → 3 = 15` so với `6 → 1 → 2 → 3 = 12`, `5 → 4 → 1 → 6 = 16` → đáp án `16`.

**Bài 3.** Mức 1: khung giống `dfs` của Cách 2. Mức 2: `total = 1` rồi cộng `dfs(x, y)` cho mỗi hàng xóm hợp lệ. Mức 3: nhớ khôi phục trước khi `return total`.

**Bài 4.** Mức 1: `visited = set()`. Mức 2: điều kiện đi tiếp `grid[x][y] and (x, y) not in visited`. Mức 3: `visited.add` trước khi đệ quy, `visited.remove` sau.

**Bài 5.** Mức 1: thêm tham số `steps_left`. Mức 2: khi `steps_left == 1` thì không đi tiếp nữa. Mức 3: vẫn phải thử mọi ô bắt đầu.

## 19. Đáp án tham khảo

In [25]:
# Bài 1, 2
print(Solution().getMaximumGold([[1, 2], [0, 3]]))
assert Solution().getMaximumGold([[1, 2], [0, 3]]) == 6
g2 = [[5, 0, 3], [4, 1, 2], [0, 6, 0]]
print(best_path(g2))
assert Solution().getMaximumGold([row[:] for row in g2]) == 16 == gold_brute(g2)

6
(16, [(0, 0), (1, 0), (1, 1), (2, 1)])


In [26]:
def count_paths(grid):
    grid = [row[:] for row in grid]
    m, n = len(grid), len(grid[0])

    def dfs(r, c):
        g = grid[r][c]
        grid[r][c] = 0
        total = 1
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and grid[x][y]:
                total += dfs(x, y)
        grid[r][c] = g
        return total

    return sum(dfs(r, c) for r in range(m) for c in range(n) if grid[r][c])


def gold_visited(grid):
    m, n = len(grid), len(grid[0])
    visited = set()

    def dfs(r, c):
        visited.add((r, c))
        best = 0
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and grid[x][y] and (x, y) not in visited:
                best = max(best, dfs(x, y))
        visited.remove((r, c))
        return grid[r][c] + best

    return max((dfs(r, c) for r in range(m) for c in range(n) if grid[r][c]), default=0)


def gold_limited(grid, L):
    grid = [row[:] for row in grid]
    m, n = len(grid), len(grid[0])

    def dfs(r, c, steps_left):
        g = grid[r][c]
        if steps_left == 1:
            return g
        grid[r][c] = 0
        best = 0
        for dr, dc in DIRS:
            x, y = r + dr, c + dc
            if 0 <= x < m and 0 <= y < n and grid[x][y]:
                best = max(best, dfs(x, y, steps_left - 1))
        grid[r][c] = g
        return g + best

    return max((dfs(r, c, L) for r in range(m) for c in range(n) if grid[r][c]), default=0)


def brute_count_paths(grid):
    cells = [(r, c) for r in range(len(grid)) for c in range(len(grid[0])) if grid[r][c]]
    return sum(all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 for a, b in zip(p, p[1:]))
               for k in range(1, len(cells) + 1) for p in permutations(cells, k))


def brute_limited(grid, L):
    cells = [(r, c) for r in range(len(grid)) for c in range(len(grid[0])) if grid[r][c]]
    best = 0
    for k in range(1, min(L, len(cells)) + 1):
        for p in permutations(cells, k):
            if all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 for a, b in zip(p, p[1:])):
                best = max(best, sum(grid[r][c] for r, c in p))
    return best


assert count_paths([[1, 1]]) == 4
assert gold_visited([[0, 6, 0], [5, 8, 7], [0, 9, 0]]) == 24
assert gold_limited(examples[0][0], 2) == 17
assert gold_limited(examples[1][0], 3) == 20
rng = random.Random(42)
for _ in range(100):
    m, n = rng.randint(1, 3), rng.randint(1, 3)
    g = [[rng.choice([0, rng.randint(1, 9)]) for _ in range(n)] for _ in range(m)]
    L = rng.randint(1, 5)
    assert count_paths(g) == brute_count_paths(g)
    assert gold_visited(g) == gold_backtrack(g)
    assert gold_limited(g, L) == brute_limited(g, L)
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
def dfs(r, c):
    g = grid[r][c]
    grid[r][c] = 0                       # đánh dấu
    best = 0
    for x, y in ((r+1,c), (r-1,c), (r,c+1), (r,c-1)):
        if 0 <= x < m and 0 <= y < n and grid[x][y]:
            best = max(best, dfs(x, y))
    grid[r][c] = g                       # KHÔI PHỤC
    return g + best

ans = max((dfs(r, c) for r in range(m) for c in range(n) if grid[r][c]), default=0)
```

**Checklist:**
- [ ] Thử **mọi** ô có vàng làm điểm bắt đầu?
- [ ] Khôi phục ô trước khi `return`?
- [ ] Kiểm tra biên trước khi đọc `grid[x][y]` (chỉ số âm không báo lỗi!)?
- [ ] Lưới không có vàng trả `0` (dùng `default=0`)?
- [ ] Có dừng sớm khi `ans == tổng vàng` (phòng test lưới đầy)?
- [ ] Không tham lam theo ô lớn nhất?